<a href="https://colab.research.google.com/github/bordeauxnate-oss/ThinkPythonAssignments/blob/main/Week15/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install widgets (only needed once in Colab)
!pip install ipywidgets

import sqlite3
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------
# Database Setup
# ---------------------------
conn = sqlite3.connect("combat_tracker.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS combatants (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    hp INTEGER,
    initiative INTEGER
)
""")
conn.commit()

# ---------------------------
# Core Logic
# ---------------------------
def add_combatant(name, hp, initiative):
    cursor.execute("INSERT INTO combatants (name, hp, initiative) VALUES (?, ?, ?)",
                   (name, hp, initiative))
    conn.commit()
    update_display()

def get_combatants():
    cursor.execute("SELECT id, name, hp, initiative FROM combatants ORDER BY initiative DESC")
    return cursor.fetchall()

def update_hp(combatant_id, change):
    cursor.execute("UPDATE combatants SET hp = hp + ? WHERE id = ?", (change, combatant_id))
    conn.commit()
    update_display()

def clear_combat():
    cursor.execute("DELETE FROM combatants")
    conn.commit()
    update_display()

# ---------------------------
# UI Elements
# ---------------------------
name_input = widgets.Text(description="Name:")
hp_input = widgets.IntText(description="HP:")
init_input = widgets.IntText(description="Initiative:")

add_button = widgets.Button(description="Add Combatant", button_style='success')
clear_button = widgets.Button(description="Clear Combat", button_style='danger')

output = widgets.Output()

# ---------------------------
# Display Function
# ---------------------------
def update_display():
    with output:
        clear_output()
        combatants = get_combatants()

        if not combatants:
            print("No combatants yet.")
            return

        for cid, name, hp, init in combatants:

            # Input for custom HP change
            hp_change_input = widgets.IntText(
                value=0,
                description="Amount:",
                layout=widgets.Layout(width='150px')
            )

            # Buttons
            dmg_button = widgets.Button(description="Damage", button_style='warning')
            heal_button = widgets.Button(description="Heal", button_style='info')

            # Button actions
            def deal_damage(b, cid=cid, input_box=hp_change_input):
                update_hp(cid, -abs(input_box.value))

            def heal(b, cid=cid, input_box=hp_change_input):
                update_hp(cid, abs(input_box.value))

            dmg_button.on_click(deal_damage)
            heal_button.on_click(heal)

            # Display row
            display(widgets.HBox([
                widgets.Label(f"{name} | HP: {hp} | Init: {init}"),
                hp_change_input,
                dmg_button,
                heal_button
            ]))

# ---------------------------
# Button Actions
# ---------------------------
def on_add_clicked(b):
    add_combatant(name_input.value, hp_input.value, init_input.value)
    name_input.value = ""
    hp_input.value = 0
    init_input.value = 0

add_button.on_click(on_add_clicked)
clear_button.on_click(lambda b: clear_combat())

# ---------------------------
# Layout
# ---------------------------
display(widgets.VBox([
    name_input,
    hp_input,
    init_input,
    add_button,
    clear_button,
    output
]))

update_display()